**COMMAND:** `ps aux | grep nginx`

**PURPOSE (support):** `ps` alone dumps every process on the box. Piping it through `grep` filters that firehose down to just the service you're investigating — e.g. "is nginx even running, and if so, how many worker processes does it have?"

**SYNTAX:** `ps aux | grep pattern`

**FLAGS (the `aux` part):**

|Flag|Meaning|
|---|---|
|`a`|show processes for **all** users, not just yours|
|`u`|display in **user-oriented** format (adds USER, %CPU, %MEM, etc.)|
|`x`|include processes **without a controlling terminal** (daemons, services)|

Note: no dash before `aux` — that's the BSD-style syntax (vs `ps -ef` which is the UNIX-style equivalent, slightly different columns but same idea).

**EXAMPLE:**

In [ ]:
$ ps aux | grep nginx
root      1122  0.0  0.1  55432  3200 ?        Ss   09:01   0:00 nginx: master process /usr/sbin/nginx
www-data  1123  0.0  0.2  55980  4100 ?        S    09:01   0:00 nginx: worker process
www-data  1124  0.0  0.2  55980  4088 ?        S    09:01   0:00 nginx: worker process
dennie    2044  0.0  0.0  17652   932 pts/0    S+   10:15   0:00 grep --color=auto nginx

**READ THE OUTPUT — column by column:**

|Column|Meaning|What to look for|
|---|---|---|
|USER|who owns the process|`root` = master process (needs privileged port 80/443); `www-data` = unprivileged workers — correct separation of privilege|
|PID|process ID|needed for `kill`, or to check `/proc/PID/`|
|%CPU / %MEM|current usage|a worker pegged at 100% CPU = stuck request or infinite loop, worth investigating|
|STAT|process state|`S` = sleeping (normal, idle waiting for work), `R` = running, `Z` = zombie (defunct, parent hasn't reaped it), `D` = uninterruptible sleep (often stuck I/O — bad sign if persistent)|
|START|when it launched|if nginx "just crashed and restarted," a very recent START time on the master is a clue|
|TIME|actual **CPU time consumed**, not wall-clock uptime — usually near 0:00 for a healthy idle service||
|COMMAND|the actual command line|tells you master vs worker, or full path/args if truncated|

**The self-match gotcha you're seeing:** that last line, `grep --color=auto nginx`, is `grep` matching its own process name in the process list — because by the time `ps` runs, your `grep nginx` command is itself a running process containing the string "nginx". Cosmetic annoyance, but worth recognizing so you don't panic-count it as a real nginx process.

**Standard fix:** `ps aux | grep [n]ginx` — wrapping the first letter in brackets means grep is now searching for the pattern `nginx` but its own command line reads `grep [n]ginx`, which does **not** match the regex `[n]ginx` literally, so it excludes itself. Clever old trick, but honestly this is exactly the itch `pgrep` scratches without the regex trick.

**TRY NOW:**

In [ ]:
dennie@HP:~/linux-tut$ ps aux | grep bash
dennie       457  0.0  0.0   6080  5284 pts/1    S+   03:48   0:00 -bash
dennie      1556  0.1  0.0   6204  5448 pts/4    Ss   03:55   0:00 /bin/bash --init-file /home/dennie/.vscode-server/bin/a44adf7f53e00964ab890f9f8758a334f1fc15bc/out/vs/workbench/contrib/terminal/common/scripts/shellIntegration-bash.sh
dennie      1758  0.0  0.0   4096  2164 pts/4    S+   03:56   0:00 grep --color=auto bash

In [ ]:
dennie@HP:~/linux-tut$ ps aux | grep [b]ash
dennie       457  0.0  0.0   6080  5284 pts/1    S+   03:48   0:00 -bash
dennie      1556  0.0  0.0   6204  5452 pts/4    Ss   03:55   0:00 /bin/bash --init-file /home/dennie/.vscode-server/bin/a44adf7f53e00964ab890f9f8758a334f1fc15bc/out/vs/workbench/contrib/terminal/common/scripts/shellIntegration-bash.sh
dennie      1955  0.0  0.0   4760  3492 pts/0    S+   03:57   0:00 /bin/bash /home/dennie/.vscode-server/bin/a44adf7f53e00964ab890f9f8758a334f1fc15bc/out/vs/base/node/cpuUsage.sh 1556

In [ ]:
-c '[b]ash'
2

Compare the three outputs: the first shows a `grep --color=auto bash` self-match line, the second (bracket trick) excludes it, and the third gives you just a count. If you have another real service running — try `ps aux | grep ssh` or `ps aux | grep code` (the VS Code server process WSL spawns) for a closer analogue to nginx.

**TICKET SCENARIO:** "Users report the website is down." First move: SSH into the box and run `ps aux | grep nginx`. Three possible outcomes:

1. No output at all (besides the grep line itself) → nginx isn't running → `systemctl status nginx` next.
2. Master process (`root`) present but zero workers → config issue preventing workers from spawning → check `journalctl -u nginx` or nginx error log.
3. Master + workers present but %CPU pegged near 100% on all workers → overload or stuck requests, not a crash → different diagnosis path entirely (check `ss -tlnp`, request logs).

Same command, three totally different next steps depending on what you read in the output — that's the whole game in support work.

**GOTCHA:** Don't confuse TIME (cumulative CPU seconds consumed) with how long the process has been alive (START, or use `ps -eo pid,etime,cmd` for elapsed wall-clock time). A process running for 3 days with `TIME 0:02` is normal (mostly idle); `TIME 45:12` on that same process is a red flag.